In [1]:
!pip install transformers datasets sentencepiece sacrebleu evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate
import numpy as np

In [3]:
dataset = load_dataset("cfilt/iitb-english-hindi")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [4]:
train_data = dataset['train'].select(range(20000))

validation_data = dataset['validation'].select(range(520))

test_data = dataset['test'].select(range(2000))

In [5]:
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})
{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}


In [6]:
small_train = dataset['train'].shuffle(seed=42).select(
    range(min(50000, len(dataset['train'])))
)

small_valid = dataset['validation'].shuffle(seed=42).select(
    range(min(520, len(dataset['validation'])))
)

small_test = dataset['test'].shuffle(seed=42).select(
    range(min(2000, len(dataset['test'])))
)

In [7]:
print(len(small_train))
print(len(small_valid))
print(len(small_test))

50000
520
2000


In [8]:
model_name = "facebook/nllb-200-distilled-600M"

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [10]:
model_name = "Helsinki-NLP/opus-mt-en-hi"

In [11]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [12]:
max_input_length = 128
max_target_length = 128


def preprocess_function(examples):
    inputs = [ex['hi'] for ex in examples['translation']]
    targets = [ex['en'] for ex in examples['translation']]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding='max_length'
    )

    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding='max_length'
    )

    model_inputs['labels'] = labels['input_ids']

    return model_inputs

In [13]:
train_dataset = small_train.map(preprocess_function, batched=True)
valid_dataset = small_valid.map(preprocess_function, batched=True)
test_dataset = small_test.map(preprocess_function, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [14]:
bleu_metric = evaluate.load("bleu")
ter_metric = evaluate.load("ter")
chrf_metric = evaluate.load("chrf")

In [15]:
def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[label] for label in decoded_labels]
    )

    ter = ter_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    chrf = chrf_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    return {
    "BLEU": bleu['bleu'],
    "TER": ter['score'],
    "ChrF": chrf['score']
    }

In [16]:
import transformers
print(transformers.__version__)


5.0.0


In [17]:
!pip install --upgrade transformers accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 110.6 MB/s eta 0:00:00


In [18]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_strategy="epoch"
)

In [19]:
from transformers import DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

In [20]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics
)

In [21]:
MODEL_NAME = "Helsinki-NLP/opus-mt-hi-en"

In [22]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/304M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/304M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [23]:
MAX_LENGTH = 128

def preprocess_function(examples):

    inputs = [ex['hi'] for ex in examples['translation']]
    targets = [ex['en'] for ex in examples['translation']]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [24]:
train_tokenized = train_data.map(
    preprocess_function,
    batched=True
)

validation_tokenized = validation_data.map(
    preprocess_function,
    batched=True
)

test_tokenized = test_data.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [25]:
sentence = "कल मैं दिल्ली जाऊंगा।"

inputs = tokenizer(sentence, return_tensors="pt")

outputs = model.generate(**inputs)

translation = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Hindi:", sentence)
print("English:", translation)

Hindi: कल मैं दिल्ली जाऊंगा।
English: I'll go to Delhi tomorrow.


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "Helsinki-NLP/opus-mt-hi-en"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/304M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/304M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [3]:
test_sentences = [
    "आज मुझे कॉलेज जाने में देर हो गई।",
    "भारत की संस्कृति दुनिया में प्रसिद्ध है।",
    "अगर मौसम अच्छा रहा तो हम घूमने जाएंगे।",
    "कृत्रिम बुद्धिमत्ता भविष्य को बदल सकती है।",
    "क्या आप मेरी मदद कर सकते हैं?"
]

for sentence in test_sentences:

    inputs = tokenizer(sentence, return_tensors="pt").to(device)

    outputs = model.generate(**inputs)

    translation = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Hindi:", sentence)
    print("English:", translation)
    print("-" * 60)

Hindi: आज मुझे कॉलेज जाने में देर हो गई।
English: Today I'm late to go to college.
------------------------------------------------------------
Hindi: भारत की संस्कृति दुनिया में प्रसिद्ध है।
English: India's culture is famous in the world.
------------------------------------------------------------
Hindi: अगर मौसम अच्छा रहा तो हम घूमने जाएंगे।
English: If the weather's good, we'll go.
------------------------------------------------------------
Hindi: कृत्रिम बुद्धिमत्ता भविष्य को बदल सकती है।
English: Design wisdom can change the future.
------------------------------------------------------------
Hindi: क्या आप मेरी मदद कर सकते हैं?
English: Can you help me?
------------------------------------------------------------


In [4]:
!pip install evaluate sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00


In [5]:
import evaluate

bleu = evaluate.load("sacrebleu")

predictions = [
    "India's culture is famous in the world.",
    "Can you help me?"
]

references = [
    ["Indian culture is famous worldwide."],
    ["Can you help me?"]
]

results = bleu.compute(
    predictions=predictions,
    references=references
)

print(results)

{'score': 46.79525195792235, 'counts': [9, 6, 4, 2], 'totals': [13, 11, 9, 7], 'precisions': [69.23076923076923, 54.54545454545455, 44.44444444444444, 28.571428571428573], 'bp': 1.0, 'sys_len': 13, 'ref_len': 11}


In [6]:
model.save_pretrained("final_hi_en_model")
tokenizer.save_pretrained("final_hi_en_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_hi_en_model/tokenizer_config.json',
 'final_hi_en_model/vocab.json',
 'final_hi_en_model/source.spm',
 'final_hi_en_model/target.spm',
 'final_hi_en_model/added_tokens.json')